In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sksurv.metrics import concordance_index_censored
import math
import seaborn as sns

pd.options.mode.chained_assignment = None

# 67_33

## 建立資料集

In [2]:
df = pd.read_csv("67_33_nandel_240226.csv")
#mask = (df.Derivation_or_Validation_group != 0)
# df = df.drop("Unnamed: 0",axis=1)
df.drop("Unnamed: 0",axis=1,inplace=True)
# df.drop("ETV1TDFTAF2",axis=1,inplace=True)
df["HCCdurationwks"] = df["HCCduration (wks)"]
df.drop("HCCduration (wks)",axis=1,inplace=True)
print(df.columns)


Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks'],
      dtype='object')


## PAGE-B

In [3]:
feature = ["Derivation_or_Validation_group","age1yr","sex1M0F","plateletR","HCC", "HCCdurationwks"]
PAGE_B_test = df[feature]
PAGE_B_test["age"] = PAGE_B_test["age1yr"]-1
PAGE_B_test.drop("age1yr",axis=1,inplace=True)
PAGE_B_test["plateletR"] = PAGE_B_test["plateletR"]*1000
PAGE_B_test

,Derivation_or_Validation_group,sex1M0F,plateletR,HCC,HCCdurationwks,age
0,1.0,0,67000.0,0,221,69.0
1,2.0,1,161000.0,0,594,75.0
2,1.0,1,122000.0,0,298,41.0
3,1.0,0,371000.0,0,549,47.0
4,1.0,1,152000.0,0,212,68.0
...,...,...,...,...,...,...
1587,2.0,1,238000.0,1,106,64.0
1588,2.0,1,85000.0,0,693,56.0
1589,1.0,1,88000.0,0,105,54.0
1590,1.0,1,192000.0,0,241,73.0


In [4]:
for i in range(PAGE_B_test["Derivation_or_Validation_group"].count()):
    if(PAGE_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (PAGE_B_test.loc[i,"age"]<=29 and PAGE_B_test.loc[i,"age"]>=16):
            pass
        elif(PAGE_B_test.loc[i,"age"]<=39 and PAGE_B_test.loc[i,"age"]>=30):
            score+=2
        elif(PAGE_B_test.loc[i,"age"]<=49 and PAGE_B_test.loc[i,"age"]>=40):
            score+=4
        elif(PAGE_B_test.loc[i,"age"]<=59 and PAGE_B_test.loc[i,"age"]>=50):
            score+=6
        elif(PAGE_B_test.loc[i,"age"]<=69 and PAGE_B_test.loc[i,"age"]>=60):
            score+=8
        elif(PAGE_B_test.loc[i,"age"]>=70):
            score+=10
        '''-------------------------------------------'''
        if(PAGE_B_test.loc[i,"sex1M0F"]==1):
            score+=6
        else:
            pass
        '''-------------------------------------------'''
        if(PAGE_B_test.loc[i,"plateletR"]>=200000):
            pass
        elif(PAGE_B_test.loc[i,"plateletR"]<=199999 and PAGE_B_test.loc[i,"plateletR"]>=100000):
            score+=6
        else:
            score+=9
        PAGE_B_test.loc[i,"PAGE_B_score"] = score
PAGE_B_test

,Derivation_or_Validation_group,sex1M0F,plateletR,HCC,HCCdurationwks,age,PAGE_B_score
0,1.0,0,67000.0,0,221,69.0,17.0
1,2.0,1,161000.0,0,594,75.0,22.0
2,1.0,1,122000.0,0,298,41.0,16.0
3,1.0,0,371000.0,0,549,47.0,4.0
4,1.0,1,152000.0,0,212,68.0,20.0
...,...,...,...,...,...,...,...
1587,2.0,1,238000.0,1,106,64.0,14.0
1588,2.0,1,85000.0,0,693,56.0,21.0
1589,1.0,1,88000.0,0,105,54.0,21.0
1590,1.0,1,192000.0,0,241,73.0,22.0


In [5]:
df["PAGE-B"] =  PAGE_B_test["PAGE_B_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,gpt1yr,T-bil1yrR,platelet1yrR,afp1yrR,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,25.0,1.00,40.0,3.30,4.50,0,1.0,1.0,221,17.0
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,24.0,1.00,147.0,7.46,4.10,0,2.0,0.0,594,22.0
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,29.0,1.90,134.0,3.80,4.83,0,1.0,1.0,298,16.0
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,25.0,0.50,480.0,4.17,4.55,0,1.0,1.0,549,4.0
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,45.0,1.00,165.0,3.10,4.54,0,1.0,1.0,212,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,68.0,0.80,128.0,2.90,4.00,1,2.0,1.0,106,14.0
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,31.0,1.30,110.0,6.69,3.80,0,2.0,1.0,693,21.0
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,39.0,1.29,86.0,4.33,3.58,0,1.0,1.0,105,21.0
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,33.0,1.70,136.0,2.60,4.92,0,1.0,1.0,241,22.0


In [6]:
df.to_csv("./compare/67_33ModelP_score.csv")

In [7]:
#trainset c-index test
trainmask = (PAGE_B_test["Derivation_or_Validation_group"]==1)
PAGE_B_train_test = PAGE_B_test[trainmask]
PAGE_B_train_test.loc[:,'HCC'] = PAGE_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_train_test['HCC'],
    PAGE_B_train_test['HCCdurationwks'],
    PAGE_B_train_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

0.61


In [8]:
#testset c-index test
testmask = (PAGE_B_test["Derivation_or_Validation_group"]==2)
PAGE_B_test_test = PAGE_B_test[testmask]
PAGE_B_test_test.loc[:,'HCC'] = PAGE_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_test_test['HCC'],
    PAGE_B_test_test['HCCdurationwks'],
    PAGE_B_test_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

0.619


In [9]:
allmask = (PAGE_B_test["Derivation_or_Validation_group"]!=0)
PAGE_B_all_test = PAGE_B_test[allmask]
PAGE_B_all_test.loc[:,'HCC'] = PAGE_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_all_test['HCC'],
    PAGE_B_all_test['HCCdurationwks'],
    PAGE_B_all_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

0.613


## mPAGE-B

In [10]:
feature = ["Derivation_or_Validation_group","age1yr","sex1M0F","plateletR","albuminR","HCC", "HCCdurationwks"]
mPAGE_B_test = df[feature]
mPAGE_B_test["age"] = mPAGE_B_test["age1yr"]-1
mPAGE_B_test.drop("age1yr",axis=1,inplace=True)
mPAGE_B_test

,Derivation_or_Validation_group,sex1M0F,plateletR,albuminR,HCC,HCCdurationwks,age
0,1.0,0,67.0,4.20,0,221,69.0
1,2.0,1,161.0,4.10,0,594,75.0
2,1.0,1,122.0,4.34,0,298,41.0
3,1.0,0,371.0,4.10,0,549,47.0
4,1.0,1,152.0,4.32,0,212,68.0
...,...,...,...,...,...,...,...
1587,2.0,1,238.0,3.40,1,106,64.0
1588,2.0,1,85.0,2.90,0,693,56.0
1589,1.0,1,88.0,3.90,0,105,54.0
1590,1.0,1,192.0,4.89,0,241,73.0


In [11]:
for i in range(mPAGE_B_test["Derivation_or_Validation_group"].count()):
    if(mPAGE_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (mPAGE_B_test.loc[i,"age"]<30):
            pass
        elif(mPAGE_B_test.loc[i,"age"]<40 and mPAGE_B_test.loc[i,"age"]>=30):
            score+=3
        elif(mPAGE_B_test.loc[i,"age"]<50 and mPAGE_B_test.loc[i,"age"]>=40):
            score+=5
        elif(mPAGE_B_test.loc[i,"age"]<60 and mPAGE_B_test.loc[i,"age"]>=50):
            score+=7
        elif(mPAGE_B_test.loc[i,"age"]<70 and mPAGE_B_test.loc[i,"age"]>=60):
            score+=9
        elif(mPAGE_B_test.loc[i,"age"]>=70):
            score+=11
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"sex1M0F"]==1):
            score+=2
        else:
            pass
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"plateletR"]>=250):
            pass
        elif(mPAGE_B_test.loc[i,"plateletR"]<250 and mPAGE_B_test.loc[i,"plateletR"]>=200):
            score+=2
        elif(mPAGE_B_test.loc[i,"plateletR"]<200 and mPAGE_B_test.loc[i,"plateletR"]>=150):
            score+=3
        elif(mPAGE_B_test.loc[i,"plateletR"]<150 and mPAGE_B_test.loc[i,"plateletR"]>=100):
            score+=4
        else:
            score+=5
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"albuminR"]>=4):
            pass
        elif(mPAGE_B_test.loc[i,"albuminR"]<4 and mPAGE_B_test.loc[i,"albuminR"]>=3.5):
            score+=1
        elif(mPAGE_B_test.loc[i,"albuminR"]<3.5 and mPAGE_B_test.loc[i,"albuminR"]>=3):
            score+=2
        else:
            score+=3
        mPAGE_B_test.loc[i,"mPAGE_B_score"] = score
mPAGE_B_test

,Derivation_or_Validation_group,sex1M0F,plateletR,albuminR,HCC,HCCdurationwks,age,mPAGE_B_score
0,1.0,0,67.0,4.20,0,221,69.0,14.0
1,2.0,1,161.0,4.10,0,594,75.0,16.0
2,1.0,1,122.0,4.34,0,298,41.0,11.0
3,1.0,0,371.0,4.10,0,549,47.0,5.0
4,1.0,1,152.0,4.32,0,212,68.0,14.0
...,...,...,...,...,...,...,...,...
1587,2.0,1,238.0,3.40,1,106,64.0,15.0
1588,2.0,1,85.0,2.90,0,693,56.0,17.0
1589,1.0,1,88.0,3.90,0,105,54.0,15.0
1590,1.0,1,192.0,4.89,0,241,73.0,16.0


In [12]:
df["mPAGE-B"] =  mPAGE_B_test["mPAGE_B_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,T-bil1yrR,platelet1yrR,afp1yrR,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,1.00,40.0,3.30,4.50,0,1.0,1.0,221,17.0,14.0
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,1.00,147.0,7.46,4.10,0,2.0,0.0,594,22.0,16.0
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,1.90,134.0,3.80,4.83,0,1.0,1.0,298,16.0,11.0
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,0.50,480.0,4.17,4.55,0,1.0,1.0,549,4.0,5.0
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,1.00,165.0,3.10,4.54,0,1.0,1.0,212,20.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,0.80,128.0,2.90,4.00,1,2.0,1.0,106,14.0,15.0
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,1.30,110.0,6.69,3.80,0,2.0,1.0,693,21.0,17.0
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,1.29,86.0,4.33,3.58,0,1.0,1.0,105,21.0,15.0
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,1.70,136.0,2.60,4.92,0,1.0,1.0,241,22.0,16.0


In [13]:
df.to_csv("./compare/67_33ModelmP_score.csv")

In [14]:
#trainset c-index test
trainmask = (mPAGE_B_test["Derivation_or_Validation_group"]==1)
mPAGE_B_train_test = mPAGE_B_test[trainmask]
mPAGE_B_train_test.loc[:,'HCC'] = mPAGE_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_train_test['HCC'],
    mPAGE_B_train_test['HCCdurationwks'],
    mPAGE_B_train_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

0.656


In [15]:
#testset c-index test
testmask = (mPAGE_B_test["Derivation_or_Validation_group"]==2)
mPAGE_B_test_test = mPAGE_B_test[testmask]
mPAGE_B_test_test.loc[:,'HCC'] = mPAGE_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_test_test['HCC'],
    mPAGE_B_test_test['HCCdurationwks'],
    mPAGE_B_test_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

0.663


In [16]:
allmask = (mPAGE_B_test["Derivation_or_Validation_group"]!=0)
mPAGE_B_all_test = mPAGE_B_test[allmask]
mPAGE_B_all_test.loc[:,'HCC'] = mPAGE_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_all_test['HCC'],
    mPAGE_B_all_test['HCCdurationwks'],
    mPAGE_B_all_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

0.659


## REACH-B

In [17]:
feature = ["Derivation_or_Validation_group","sex1M0F","age1yr","gpt","HBeAg","logHBVDNA","HCC", "HCCdurationwks"]
REACH_B_test = df[feature]
REACH_B_test["age"] = REACH_B_test["age1yr"]-1
REACH_B_test.drop("age1yr",axis=1,inplace=True)
REACH_B_test

,Derivation_or_Validation_group,sex1M0F,gpt,HBeAg,logHBVDNA,HCC,HCCdurationwks,age
0,1.0,0,31,0.0,4.77,0,221,69.0
1,2.0,1,73,1.0,7.83,0,594,75.0
2,1.0,1,72,0.0,6.93,0,298,41.0
3,1.0,0,38,0.0,5.47,0,549,47.0
4,1.0,1,166,1.0,6.52,0,212,68.0
...,...,...,...,...,...,...,...,...
1587,2.0,1,427,0.0,3.88,1,106,64.0
1588,2.0,1,236,0.0,3.52,0,693,56.0
1589,1.0,1,44,0.0,5.36,0,105,54.0
1590,1.0,1,77,0.0,6.31,0,241,73.0


In [18]:
for i in range(REACH_B_test["Derivation_or_Validation_group"].count()):
    REACH_B_test.loc[i,"HBVDNA"] = math.pow(10,(REACH_B_test.loc[i,"logHBVDNA"]))*5.6
REACH_B_test

,Derivation_or_Validation_group,sex1M0F,gpt,HBeAg,logHBVDNA,HCC,HCCdurationwks,age,HBVDNA
0,1.0,0,31,0.0,4.77,0,221,69.0,3.297524e+05
1,2.0,1,73,1.0,7.83,0,594,75.0,3.786065e+08
2,1.0,1,72,0.0,6.93,0,298,41.0,4.766373e+07
3,1.0,0,38,0.0,5.47,0,549,47.0,1.652677e+06
4,1.0,1,166,1.0,6.52,0,212,68.0,1.854334e+07
...,...,...,...,...,...,...,...,...,...
1587,2.0,1,427,0.0,3.88,1,106,64.0,4.248034e+04
1588,2.0,1,236,0.0,3.52,0,693,56.0,1.854334e+04
1589,1.0,1,44,0.0,5.36,0,105,54.0,1.282886e+06
1590,1.0,1,77,0.0,6.31,0,241,73.0,1.143373e+07


In [19]:
for i in range(REACH_B_test["Derivation_or_Validation_group"].count()):
    if(REACH_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (REACH_B_test.loc[i,"age"]<35): #評分表上面從30歲開始，但30-34歲還是0分，所以我把35歲以下全評成0分
            pass
        elif(REACH_B_test.loc[i,"age"]<40 and REACH_B_test.loc[i,"age"]>=35):
            score+=1
        elif(REACH_B_test.loc[i,"age"]<45 and REACH_B_test.loc[i,"age"]>=40):
            score+=2
        elif(REACH_B_test.loc[i,"age"]<50 and REACH_B_test.loc[i,"age"]>=45):
            score+=3
        elif(REACH_B_test.loc[i,"age"]<55 and REACH_B_test.loc[i,"age"]>=50):
            score+=4
        elif(REACH_B_test.loc[i,"age"]<60 and REACH_B_test.loc[i,"age"]>=55):
            score+=5
        elif(REACH_B_test.loc[i,"age"]>=60): #評分表最多到65歲，60-65歲是6分，所以我把60歲以上全評成6分
            score+=6
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"sex1M0F"]==1):
            score+=2
        else:
            pass
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"gpt"]<15):
            pass
        elif(REACH_B_test.loc[i,"gpt"]>=15 and REACH_B_test.loc[i,"gpt"]<45):
            score+=1
        else:
            score+=2
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"HBeAg"]==1):
            score+=2
        else:
            pass
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"HBVDNA"]<10000):
            pass
        elif(REACH_B_test.loc[i,"HBVDNA"]>=10000 and REACH_B_test.loc[i,"HBVDNA"]<100000):
            score+=3
        elif(REACH_B_test.loc[i,"HBVDNA"]>=100000 and REACH_B_test.loc[i,"HBVDNA"]<1000000):
            score+=5
        else:
            score+=4
        REACH_B_test.loc[i,"REACH_B_score"] = score
mPAGE_B_test

,Derivation_or_Validation_group,sex1M0F,plateletR,albuminR,HCC,HCCdurationwks,age,mPAGE_B_score
0,1.0,0,67.0,4.20,0,221,69.0,14.0
1,2.0,1,161.0,4.10,0,594,75.0,16.0
2,1.0,1,122.0,4.34,0,298,41.0,11.0
3,1.0,0,371.0,4.10,0,549,47.0,5.0
4,1.0,1,152.0,4.32,0,212,68.0,14.0
...,...,...,...,...,...,...,...,...
1587,2.0,1,238.0,3.40,1,106,64.0,15.0
1588,2.0,1,85.0,2.90,0,693,56.0,17.0
1589,1.0,1,88.0,3.90,0,105,54.0,15.0
1590,1.0,1,192.0,4.89,0,241,73.0,16.0


In [20]:
df["REACH-B"] =  REACH_B_test["REACH_B_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,platelet1yrR,afp1yrR,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,40.0,3.30,4.50,0,1.0,1.0,221,17.0,14.0,12.0
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,147.0,7.46,4.10,0,2.0,0.0,594,22.0,16.0,16.0
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,134.0,3.80,4.83,0,1.0,1.0,298,16.0,11.0,10.0
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,480.0,4.17,4.55,0,1.0,1.0,549,4.0,5.0,8.0
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,165.0,3.10,4.54,0,1.0,1.0,212,20.0,14.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,128.0,2.90,4.00,1,2.0,1.0,106,14.0,15.0,13.0
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,110.0,6.69,3.80,0,2.0,1.0,693,21.0,17.0,12.0
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,86.0,4.33,3.58,0,1.0,1.0,105,21.0,15.0,11.0
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,136.0,2.60,4.92,0,1.0,1.0,241,22.0,16.0,14.0


In [21]:
df.to_csv("./compare/67_33ModelmPR_score.csv")

In [22]:
#trainset c-index test
trainmask = (REACH_B_test["Derivation_or_Validation_group"]==1)
REACH_B_train_test = REACH_B_test[trainmask]
REACH_B_train_test.loc[:,'HCC'] = REACH_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_train_test['HCC'],
    REACH_B_train_test['HCCdurationwks'],
    REACH_B_train_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

0.623


In [23]:
#testset c-index test
testmask = (REACH_B_test["Derivation_or_Validation_group"]==2)
REACH_B_test_test = REACH_B_test[testmask]
REACH_B_test_test.loc[:,'HCC'] = REACH_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_test_test['HCC'],
    REACH_B_test_test['HCCdurationwks'],
    REACH_B_test_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

0.621


In [24]:
allmask = (REACH_B_test["Derivation_or_Validation_group"]!=0)
REACH_B_all_test = REACH_B_test[allmask]
REACH_B_all_test.loc[:,'HCC'] = REACH_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_all_test['HCC'],
    REACH_B_all_test['HCCdurationwks'],
    REACH_B_all_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

0.622


## PLAN-B

In [25]:
PLAN_B = df.copy()
PLAN_B_test = PLAN_B[["Derivation_or_Validation_group","age1yr"]]
PLAN_B_test["age"] = PLAN_B_test["age1yr"]-1
PLAN_B_test.drop("age1yr",axis=1,inplace=True)
PLAN_B_test["male"] = PLAN_B["sex1M0F"]
PLAN_B_test["e_t"] = PLAN_B["ETV1TDFTAF2"]
PLAN_B_test["lc"] = 1
PLAN_B_test["plt"] = PLAN_B["plateletR"]
PLAN_B_test["alb"] = PLAN_B["albuminR"]
PLAN_B_test["tbil"] = PLAN_B["t-bil"]
PLAN_B_test["alt"] = PLAN_B["gpt"]
PLAN_B_test["dna"] = PLAN_B["logHBVDNA"]
PLAN_B_test["hbeag"] = PLAN_B["HBeAg"]
PLAN_B_test["HCC"] = PLAN_B["HCC"]
PLAN_B_test["HCCdurationwks"] = PLAN_B["HCCdurationwks"]
PLAN_B_test["codenumber"] = PLAN_B["codenumber"]
PLAN_B_test

,Derivation_or_Validation_group,age,male,e_t,lc,plt,alb,tbil,alt,dna,hbeag,HCC,HCCdurationwks,codenumber
0,1.0,69.0,0,1,1,67.0,4.20,1.11,31,4.77,0.0,0,221,1583
1,2.0,75.0,1,1,1,161.0,4.10,0.80,73,7.83,1.0,0,594,537
2,1.0,41.0,1,2,1,122.0,4.34,1.00,72,6.93,0.0,0,298,930
3,1.0,47.0,0,2,1,371.0,4.10,0.20,38,5.47,0.0,0,549,113
4,1.0,68.0,1,2,1,152.0,4.32,0.90,166,6.52,1.0,0,212,1049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,2.0,64.0,1,1,1,238.0,3.40,0.60,427,3.88,0.0,1,106,130
1588,2.0,56.0,1,1,1,85.0,2.90,17.00,236,3.52,0.0,0,693,165
1589,1.0,54.0,1,1,1,88.0,3.90,0.80,44,5.36,0.0,0,105,1462
1590,1.0,73.0,1,1,1,192.0,4.89,1.30,77,6.31,0.0,0,241,981


In [26]:
mask1 = (PLAN_B_test.Derivation_or_Validation_group==1)
mask2 = (PLAN_B_test.Derivation_or_Validation_group==2)
trainman = PLAN_B_test[mask1].reset_index(drop=True)
print("推導隊列有",trainman["age"].count(),"人")
testman = PLAN_B_test[mask2].reset_index(drop=True)
print("驗證隊列有",testman["age"].count(),"人")
print("推導隊列佔",trainman["age"].count()/(trainman["age"].count()+testman["age"].count()))

trainHCClist = (trainman.HCC==1)
trainHCC = trainman[trainHCClist]
print("訓練資料集數量為",len(trainman),"，其中罹患肝癌患者有",len(trainHCC),"人，佔訓練集比例為",len(trainHCC)/len(trainman))
testHCClist = (testman.HCC==1)
testHCC = testman[testHCClist]
print("測試資料集數量為",len(testman),"，其中罹患肝癌患者有",len(testHCC),"人，佔測試集比例為",len(testHCC)/len(testman))

推導隊列有 1066 人
驗證隊列有 526 人
推導隊列佔 0.6695979899497487
訓練資料集數量為 1066 ，其中罹患肝癌患者有 212 人，佔訓練集比例為 0.19887429643527205
測試資料集數量為 526 ，其中罹患肝癌患者有 105 人，佔測試集比例為 0.19961977186311788


In [27]:
PLAN_B_test_order = PLAN_B_test[["Derivation_or_Validation_group","male","age","e_t","lc","plt","alb","tbil","alt","dna","hbeag","HCC","HCCdurationwks","codenumber"]]
PLAN_B_test_order

,Derivation_or_Validation_group,male,age,e_t,lc,plt,alb,tbil,alt,dna,hbeag,HCC,HCCdurationwks,codenumber
0,1.0,0,69.0,1,1,67.0,4.20,1.11,31,4.77,0.0,0,221,1583
1,2.0,1,75.0,1,1,161.0,4.10,0.80,73,7.83,1.0,0,594,537
2,1.0,1,41.0,2,1,122.0,4.34,1.00,72,6.93,0.0,0,298,930
3,1.0,0,47.0,2,1,371.0,4.10,0.20,38,5.47,0.0,0,549,113
4,1.0,1,68.0,2,1,152.0,4.32,0.90,166,6.52,1.0,0,212,1049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,2.0,1,64.0,1,1,238.0,3.40,0.60,427,3.88,0.0,1,106,130
1588,2.0,1,56.0,1,1,85.0,2.90,17.00,236,3.52,0.0,0,693,165
1589,1.0,1,54.0,1,1,88.0,3.90,0.80,44,5.36,0.0,0,105,1462
1590,1.0,1,73.0,1,1,192.0,4.89,1.30,77,6.31,0.0,0,241,981


In [28]:
!wget https://github.com/vitaldb/hbvhcc/raw/main/model.json

--2024-03-12 14:24:17--  https://github.com/vitaldb/hbvhcc/raw/main/model.json
正在查找主機 github.com (github.com)... 20.27.177.113
正在連接 github.com (github.com)|20.27.177.113|:443... 連上了。
已送出 HTTP 要求，正在等候回應... 301 Moved Permanently
位置: https://github.com/vitaldb/planb/raw/main/model.json [跟隨至新的 URL]
--2024-03-12 14:24:17--  https://github.com/vitaldb/planb/raw/main/model.json
繼續使用和 github.com:443 的連線。
已送出 HTTP 要求，正在等候回應... 302 Found
位置: https://raw.githubusercontent.com/vitaldb/planb/main/model.json [跟隨至新的 URL]
--2024-03-12 14:24:18--  https://raw.githubusercontent.com/vitaldb/planb/main/model.json
正在查找主機 raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
正在連接 raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... 連上了。
已送出 HTTP 要求，正在等候回應... 200 OK
長度: 60814 (59K) [text/plain]
儲存到：「model.json」

model.json          100%[===================>]  59.39K  --.-KB/s  於 0.06s      

2024-03-12 14:24:18 (993 KB/s) - 已儲

In [29]:
model = xgb.Booster()
model.load_model('model.json')
inxgb = (PLAN_B_test_order["Derivation_or_Validation_group"]!=0)
PLAN_B_test = PLAN_B_test_order[inxgb] #後面不會用到，取代掉沒關係
X = PLAN_B_test.drop(["Derivation_or_Validation_group","HCC","HCCdurationwks","codenumber"],axis=1).astype(float).values
dtest_reg = xgb.DMatrix(X, PLAN_B_test['HCC'])
PLAN_B_test['PLAN_B_score'] = model.predict(dtest_reg)
PLAN_B_test

,Derivation_or_Validation_group,male,age,e_t,lc,plt,alb,tbil,alt,dna,hbeag,HCC,HCCdurationwks,codenumber,PLAN_B_score
0,1.0,0,69.0,1,1,67.0,4.20,1.11,31,4.77,0.0,0,221,1583,0.355658
1,2.0,1,75.0,1,1,161.0,4.10,0.80,73,7.83,1.0,0,594,537,0.463228
2,1.0,1,41.0,2,1,122.0,4.34,1.00,72,6.93,0.0,0,298,930,0.253839
3,1.0,0,47.0,2,1,371.0,4.10,0.20,38,5.47,0.0,0,549,113,0.197364
4,1.0,1,68.0,2,1,152.0,4.32,0.90,166,6.52,1.0,0,212,1049,0.388369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,2.0,1,64.0,1,1,238.0,3.40,0.60,427,3.88,0.0,1,106,130,0.267926
1588,2.0,1,56.0,1,1,85.0,2.90,17.00,236,3.52,0.0,0,693,165,0.389855
1589,1.0,1,54.0,1,1,88.0,3.90,0.80,44,5.36,0.0,0,105,1462,0.447126
1590,1.0,1,73.0,1,1,192.0,4.89,1.30,77,6.31,0.0,0,241,981,0.358544


In [30]:
merge = PLAN_B_test[["codenumber","PLAN_B_score"]]
df = pd.merge(df,merge,how='outer')
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,afp1yrR,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,3.30,4.50,0,1.0,1.0,221,17.0,14.0,12.0,0.355658
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,7.46,4.10,0,2.0,0.0,594,22.0,16.0,16.0,0.463228
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,3.80,4.83,0,1.0,1.0,298,16.0,11.0,10.0,0.253839
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,4.17,4.55,0,1.0,1.0,549,4.0,5.0,8.0,0.197364
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,3.10,4.54,0,1.0,1.0,212,20.0,14.0,16.0,0.388369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,2.90,4.00,1,2.0,1.0,106,14.0,15.0,13.0,0.267926
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,6.69,3.80,0,2.0,1.0,693,21.0,17.0,12.0,0.389855
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,4.33,3.58,0,1.0,1.0,105,21.0,15.0,11.0,0.447126
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,2.60,4.92,0,1.0,1.0,241,22.0,16.0,14.0,0.358544


In [31]:
df.to_csv("./compare/67_33ModelmPRL_score.csv")

In [32]:
#trainset c-index test
trainmask = (PLAN_B_test["Derivation_or_Validation_group"]==1)
PLAN_B_train = PLAN_B_test[trainmask]
PLAN_B_train.loc[:,'HCC'] = PLAN_B_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_train['HCC'],
    PLAN_B_train['HCCdurationwks'],
    PLAN_B_train["PLAN_B_score"], 
)
print(round(cindex[0], 3))

0.656


In [33]:
#testset c-index test
testmask = (PLAN_B_test["Derivation_or_Validation_group"]==2)
PLAN_B_test_test = PLAN_B_test[testmask]
PLAN_B_test_test.loc[:,'HCC'] = PLAN_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_test_test['HCC'],
    PLAN_B_test_test['HCCdurationwks'],
    PLAN_B_test_test["PLAN_B_score"], 
)
print(round(cindex[0], 3))

0.632


In [34]:
allmask = (PLAN_B_test["Derivation_or_Validation_group"]!=0)
PLAN_B_all = PLAN_B_test[allmask]
PLAN_B_all.loc[:,'HCC'] = PLAN_B_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_all['HCC'],
    PLAN_B_all['HCCdurationwks'],
    PLAN_B_all["PLAN_B_score"], 
)
print(round(cindex[0], 3))

0.649


## PLAN-B-Retrain

In [35]:
PLAN_B = df.copy()
PLAN_B_test = PLAN_B[["Derivation_or_Validation_group","age1yr"]]
PLAN_B_test["age"] = PLAN_B_test["age1yr"]-1
PLAN_B_test.drop("age1yr",axis=1,inplace=True)
PLAN_B_test["male"] = PLAN_B["sex1M0F"]
PLAN_B_test["e_t"] = PLAN_B["ETV1TDFTAF2"]
PLAN_B_test["lc"] = 1
PLAN_B_test["plt"] = PLAN_B["plateletR"]
PLAN_B_test["alb"] = PLAN_B["albuminR"]
PLAN_B_test["tbil"] = PLAN_B["t-bil"]
PLAN_B_test["alt"] = PLAN_B["gpt"]
PLAN_B_test["dna"] = PLAN_B["logHBVDNA"]
PLAN_B_test["hbeag"] = PLAN_B["HBeAg"]
PLAN_B_test["HCC"] = PLAN_B["HCC"]
PLAN_B_test["HCCdurationwks"] = PLAN_B["HCCdurationwks"]
PLAN_B_test["codenumber"] = PLAN_B["codenumber"]
PLAN_B_test

,Derivation_or_Validation_group,age,male,e_t,lc,plt,alb,tbil,alt,dna,hbeag,HCC,HCCdurationwks,codenumber
0,1.0,69.0,0,1,1,67.0,4.20,1.11,31,4.77,0.0,0,221,1583
1,2.0,75.0,1,1,1,161.0,4.10,0.80,73,7.83,1.0,0,594,537
2,1.0,41.0,1,2,1,122.0,4.34,1.00,72,6.93,0.0,0,298,930
3,1.0,47.0,0,2,1,371.0,4.10,0.20,38,5.47,0.0,0,549,113
4,1.0,68.0,1,2,1,152.0,4.32,0.90,166,6.52,1.0,0,212,1049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,2.0,64.0,1,1,1,238.0,3.40,0.60,427,3.88,0.0,1,106,130
1588,2.0,56.0,1,1,1,85.0,2.90,17.00,236,3.52,0.0,0,693,165
1589,1.0,54.0,1,1,1,88.0,3.90,0.80,44,5.36,0.0,0,105,1462
1590,1.0,73.0,1,1,1,192.0,4.89,1.30,77,6.31,0.0,0,241,981


In [36]:
PLAN_B_test_order = PLAN_B_test[["Derivation_or_Validation_group","male","age","e_t","lc","plt","alb","tbil","alt","dna","hbeag","HCC","HCCdurationwks","codenumber"]]
PLAN_B_test_order

,Derivation_or_Validation_group,male,age,e_t,lc,plt,alb,tbil,alt,dna,hbeag,HCC,HCCdurationwks,codenumber
0,1.0,0,69.0,1,1,67.0,4.20,1.11,31,4.77,0.0,0,221,1583
1,2.0,1,75.0,1,1,161.0,4.10,0.80,73,7.83,1.0,0,594,537
2,1.0,1,41.0,2,1,122.0,4.34,1.00,72,6.93,0.0,0,298,930
3,1.0,0,47.0,2,1,371.0,4.10,0.20,38,5.47,0.0,0,549,113
4,1.0,1,68.0,2,1,152.0,4.32,0.90,166,6.52,1.0,0,212,1049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,2.0,1,64.0,1,1,238.0,3.40,0.60,427,3.88,0.0,1,106,130
1588,2.0,1,56.0,1,1,85.0,2.90,17.00,236,3.52,0.0,0,693,165
1589,1.0,1,54.0,1,1,88.0,3.90,0.80,44,5.36,0.0,0,105,1462
1590,1.0,1,73.0,1,1,192.0,4.89,1.30,77,6.31,0.0,0,241,981


In [37]:
trainmask = (PLAN_B_test_order["Derivation_or_Validation_group"]==1)
train = PLAN_B_test_order[trainmask]
print(len(train.index))

1066


In [38]:
X_train = train.drop(["HCC","HCCdurationwks","codenumber","Derivation_or_Validation_group"],axis=1)
y_train = train[["HCC","HCCdurationwks","codenumber"]]
print(X_train)

      male   age  e_t  lc    plt   alb  tbil   alt   dna  hbeag
0        0  69.0    1   1   67.0  4.20  1.11    31  4.77    0.0
2        1  41.0    2   1  122.0  4.34  1.00    72  6.93    0.0
3        0  47.0    2   1  371.0  4.10  0.20    38  5.47    0.0
4        1  68.0    2   1  152.0  4.32  0.90   166  6.52    1.0
5        1  33.0    1   1  182.0  4.00  5.40  3063  6.79    1.0
...    ...   ...  ...  ..    ...   ...   ...   ...   ...    ...
1583     1  59.0    2   1   89.0  4.61  2.30    24  1.30    0.0
1584     1  62.0    2   1  142.0  4.40  0.60    46  4.34    0.0
1586     1  40.0    2   1  261.0  3.74  0.80    56  3.67    0.0
1589     1  54.0    1   1   88.0  3.90  0.80    44  5.36    0.0
1590     1  73.0    1   1  192.0  4.89  1.30    77  6.31    0.0

[1066 rows x 10 columns]


In [39]:
print(X_train.describe)

<bound method NDFrame.describe of       male   age  e_t  lc    plt   alb  tbil   alt   dna  hbeag
0        0  69.0    1   1   67.0  4.20  1.11    31  4.77    0.0
2        1  41.0    2   1  122.0  4.34  1.00    72  6.93    0.0
3        0  47.0    2   1  371.0  4.10  0.20    38  5.47    0.0
4        1  68.0    2   1  152.0  4.32  0.90   166  6.52    1.0
5        1  33.0    1   1  182.0  4.00  5.40  3063  6.79    1.0
...    ...   ...  ...  ..    ...   ...   ...   ...   ...    ...
1583     1  59.0    2   1   89.0  4.61  2.30    24  1.30    0.0
1584     1  62.0    2   1  142.0  4.40  0.60    46  4.34    0.0
1586     1  40.0    2   1  261.0  3.74  0.80    56  3.67    0.0
1589     1  54.0    1   1   88.0  3.90  0.80    44  5.36    0.0
1590     1  73.0    1   1  192.0  4.89  1.30    77  6.31    0.0

[1066 rows x 10 columns]>


In [40]:
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
params = {
        'booster': 'gbtree',
        'objective': 'binary:logistic',
        'eta': 0.07,
        'max_depth': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'tree_method': 'exact',
        'seed': 1000,
        'scale_pos_weight':5,
        }

In [41]:
num_rounds = 50
plst = list(params.items())
model = xgb.train(plst, dtrain_reg, num_rounds)

In [42]:
testmask = (PLAN_B_test_order["Derivation_or_Validation_group"]==2)
test = PLAN_B_test_order[testmask]
X_test = test.drop(["HCC","HCCdurationwks","codenumber","Derivation_or_Validation_group"],axis=1)
y_test = test[["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

In [43]:
y_test.loc[:,'pred'] = model.predict(dtest_reg)
print(y_test)

      HCC  HCCdurationwks  codenumber      pred
1       0             594         537  0.647754
10      0             646         803  0.315870
13      0             128         998  0.536095
15      0             417        1289  0.346015
19      0             699         802  0.407705
...   ...             ...         ...       ...
1580    0              63        1630  0.794834
1585    0             448         929  0.124873
1587    1             106         130  0.686385
1588    0             693         165  0.689201
1591    1             347         436  0.461629

[526 rows x 4 columns]


In [44]:
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print(round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

0.684


In [45]:
y_train.loc[:,'pred'] = model.predict(dtrain_reg)
print(y_train)

      HCC  HCCdurationwks  codenumber      pred
0       0             221        1583  0.629871
2       0             298         930  0.310707
3       0             549         113  0.302011
4       0             212        1049  0.412258
5       0             362          42  0.128407
...   ...             ...         ...       ...
1583    0             376         334  0.149502
1584    1             165         557  0.545736
1586    0             603         102  0.217671
1589    0             105        1462  0.618932
1590    0             241         981  0.633538

[1066 rows x 4 columns]


In [46]:
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print(round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

0.8


In [47]:
model.save_model('./compare/67_33model_PLANB_retrain.json')

In [48]:
tryy = df
ttt = pd.merge(tryy,y_train[["codenumber","pred"]],on="codenumber",how="outer")
ttt

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score,pred
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,4.50,0,1.0,1.0,221,17.0,14.0,12.0,0.355658,0.629871
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,4.10,0,2.0,0.0,594,22.0,16.0,16.0,0.463228,NaN
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,4.83,0,1.0,1.0,298,16.0,11.0,10.0,0.253839,0.310707
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,4.55,0,1.0,1.0,549,4.0,5.0,8.0,0.197364,0.302011
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,4.54,0,1.0,1.0,212,20.0,14.0,16.0,0.388369,0.412258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,4.00,1,2.0,1.0,106,14.0,15.0,13.0,0.267926,NaN
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,3.80,0,2.0,1.0,693,21.0,17.0,12.0,0.389855,NaN
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,3.58,0,1.0,1.0,105,21.0,15.0,11.0,0.447126,0.618932
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,4.92,0,1.0,1.0,241,22.0,16.0,14.0,0.358544,0.633538


In [49]:
xxx = pd.merge(ttt,y_test[["codenumber","pred"]],on="codenumber",how="outer")
xxx

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score,pred_x,pred_y
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,0,1.0,1.0,221,17.0,14.0,12.0,0.355658,0.629871,NaN
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,0,2.0,0.0,594,22.0,16.0,16.0,0.463228,NaN,0.647754
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,0,1.0,1.0,298,16.0,11.0,10.0,0.253839,0.310707,NaN
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,0,1.0,1.0,549,4.0,5.0,8.0,0.197364,0.302011,NaN
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,0,1.0,1.0,212,20.0,14.0,16.0,0.388369,0.412258,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,1,2.0,1.0,106,14.0,15.0,13.0,0.267926,NaN,0.686385
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,0,2.0,1.0,693,21.0,17.0,12.0,0.389855,NaN,0.689201
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,0,1.0,1.0,105,21.0,15.0,11.0,0.447126,0.618932,NaN
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,0,1.0,1.0,241,22.0,16.0,14.0,0.358544,0.633538,NaN


In [50]:
predmerge = xxx.copy()
for i in range(predmerge["codenumber"].count()):
    if (predmerge.loc[i,"pred_x"]>=0):
        predmerge.loc[i,"PLAN_B_retrain_score"] = predmerge.loc[i,"pred_x"]
    else:
        predmerge.loc[i,"PLAN_B_retrain_score"] = predmerge.loc[i,"pred_y"]
df["PLAN_B_retrain_score"] = predmerge["PLAN_B_retrain_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,albuminR.1,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score,PLAN_B_retrain_score
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,4.50,0,1.0,1.0,221,17.0,14.0,12.0,0.355658,0.629871
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,4.10,0,2.0,0.0,594,22.0,16.0,16.0,0.463228,0.647754
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,4.83,0,1.0,1.0,298,16.0,11.0,10.0,0.253839,0.310707
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,4.55,0,1.0,1.0,549,4.0,5.0,8.0,0.197364,0.302011
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,4.54,0,1.0,1.0,212,20.0,14.0,16.0,0.388369,0.412258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,4.00,1,2.0,1.0,106,14.0,15.0,13.0,0.267926,0.686385
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,3.80,0,2.0,1.0,693,21.0,17.0,12.0,0.389855,0.689201
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,3.58,0,1.0,1.0,105,21.0,15.0,11.0,0.447126,0.618932
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,4.92,0,1.0,1.0,241,22.0,16.0,14.0,0.358544,0.633538


In [51]:
df.to_csv("./compare/67_33ModelmPRLr.csv")

## APA-B

In [52]:
df = pd.read_csv("./compare/67_33ModelmPRLr.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks', 'PAGE-B', 'mPAGE-B', 'REACH-B',
       'PLAN_B_score', 'PLAN_B_retrain_score'],
      dtype='object')

In [53]:
feature = ["Derivation_or_Validation_group","age1yr","platelet1yrR","afp1yrR","HCC", "HCCdurationwks"]
APA_B = df[feature]
APA_B

,Derivation_or_Validation_group,age1yr,platelet1yrR,afp1yrR,HCC,HCCdurationwks
0,1.0,70.0,40.0,3.30,0,221
1,2.0,76.0,147.0,7.46,0,594
2,1.0,42.0,134.0,3.80,0,298
3,1.0,48.0,480.0,4.17,0,549
4,1.0,69.0,165.0,3.10,0,212
...,...,...,...,...,...,...
1587,2.0,65.0,128.0,2.90,1,106
1588,2.0,57.0,110.0,6.69,0,693
1589,1.0,55.0,86.0,4.33,0,105
1590,1.0,74.0,136.0,2.60,0,241


In [54]:
for i in range(APA_B["Derivation_or_Validation_group"].count()):
    if(APA_B.loc[i,"Derivation_or_Validation_group"] != 0):
        score = 0
        if (APA_B.loc[i,"age1yr"]<40):
            pass
        elif(APA_B.loc[i,"age1yr"]<50 and APA_B.loc[i,"age1yr"]>=40):
            score+=1
        elif(APA_B.loc[i,"age1yr"]<60 and APA_B.loc[i,"age1yr"]>=50):
            score+=2
        elif(APA_B.loc[i,"age1yr"]<70 and APA_B.loc[i,"age1yr"]>=60):
            score+=3
        elif(APA_B.loc[i,"age1yr"]>=70):
            score+=4
        '''-----------------------------------------------------'''
        if(APA_B.loc[i,"platelet1yrR"]>=130):
            pass
        elif(APA_B.loc[i,"platelet1yrR"]<130 and APA_B.loc[i,"platelet1yrR"]>=100):
            score+=3
        else:
            score+=6
        '''-----------------------------------------------------'''
        if(APA_B.loc[i,"afp1yrR"] < 5):
            pass
        elif(APA_B.loc[i,"afp1yrR"]>=5 and APA_B.loc[i,"afp1yrR"]<=9):
            score+=2
        else:
            score+=5
        APA_B.loc[i,"APA_B_score"] = score
APA_B

,Derivation_or_Validation_group,age1yr,platelet1yrR,afp1yrR,HCC,HCCdurationwks,APA_B_score
0,1.0,70.0,40.0,3.30,0,221,10.0
1,2.0,76.0,147.0,7.46,0,594,6.0
2,1.0,42.0,134.0,3.80,0,298,1.0
3,1.0,48.0,480.0,4.17,0,549,1.0
4,1.0,69.0,165.0,3.10,0,212,3.0
...,...,...,...,...,...,...,...
1587,2.0,65.0,128.0,2.90,1,106,6.0
1588,2.0,57.0,110.0,6.69,0,693,7.0
1589,1.0,55.0,86.0,4.33,0,105,8.0
1590,1.0,74.0,136.0,2.60,0,241,4.0


In [55]:
df["APA-B"] =  APA_B["APA_B_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,HCC,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score,PLAN_B_retrain_score,APA-B
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,0,1.0,1.0,221,17.0,14.0,12.0,0.355658,0.629871,10.0
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,0,2.0,0.0,594,22.0,16.0,16.0,0.463228,0.647754,6.0
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,0,1.0,1.0,298,16.0,11.0,10.0,0.253839,0.310707,1.0
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,0,1.0,1.0,549,4.0,5.0,8.0,0.197364,0.302011,1.0
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,0,1.0,1.0,212,20.0,14.0,16.0,0.388369,0.412258,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,1,2.0,1.0,106,14.0,15.0,13.0,0.267926,0.686385,6.0
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,0,2.0,1.0,693,21.0,17.0,12.0,0.389855,0.689201,7.0
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,0,1.0,1.0,105,21.0,15.0,11.0,0.447126,0.618932,8.0
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,0,1.0,1.0,241,22.0,16.0,14.0,0.358544,0.633538,4.0


In [56]:
df.to_csv("./compare/67_33ModelmPRLrA_score.csv")

In [57]:
#trainset c-index test
trainmask = (APA_B["Derivation_or_Validation_group"]==1)
APA_B_train = APA_B[trainmask]
APA_B_train.loc[:,'HCC'] = APA_B_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_train['HCC'],
    APA_B_train['HCCdurationwks'],
    APA_B_train["APA_B_score"], 
)
print(round(cindex[0], 3))

0.663


In [58]:
#testset c-index test
testmask = (APA_B["Derivation_or_Validation_group"]==2)
APA_B_test = APA_B[testmask]
APA_B_test.loc[:,'HCC'] = APA_B_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_test['HCC'],
    APA_B_test['HCCdurationwks'],
    APA_B_test["APA_B_score"], 
)
print(round(cindex[0], 3))

0.715


In [59]:
allmask = (APA_B["Derivation_or_Validation_group"]!=0)
APA_B_all = APA_B[allmask]
APA_B_all.loc[:,'HCC'] = APA_B_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_all['HCC'],
    APA_B_all['HCCdurationwks'],
    APA_B_all["APA_B_score"], 
)
print(round(cindex[0], 3))

0.68


## CU-HCC

In [60]:
df = pd.read_csv("./compare/67_33ModelmPRLrA_score.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks', 'PAGE-B', 'mPAGE-B', 'REACH-B',
       'PLAN_B_score', 'PLAN_B_retrain_score', 'APA-B'],
      dtype='object')

In [61]:
feature = ["Derivation_or_Validation_group","age1yr","albuminR","t-bil","logHBVDNA","HCC", "HCCdurationwks"]
CU_HCC = df[feature]
CU_HCC["age"] = CU_HCC["age1yr"]-1
CU_HCC.drop("age1yr",axis=1,inplace=True)
CU_HCC["Cirrhosis"] = 1
CU_HCC["albuminR"] = CU_HCC["albuminR"]*10
CU_HCC["t-bil"] = CU_HCC["t-bil"]*17.1
CU_HCC

,Derivation_or_Validation_group,albuminR,t-bil,logHBVDNA,HCC,HCCdurationwks,age,Cirrhosis
0,1.0,42.0,18.981,4.77,0,221,69.0,1
1,2.0,41.0,13.680,7.83,0,594,75.0,1
2,1.0,43.4,17.100,6.93,0,298,41.0,1
3,1.0,41.0,3.420,5.47,0,549,47.0,1
4,1.0,43.2,15.390,6.52,0,212,68.0,1
...,...,...,...,...,...,...,...,...
1587,2.0,34.0,10.260,3.88,1,106,64.0,1
1588,2.0,29.0,290.700,3.52,0,693,56.0,1
1589,1.0,39.0,13.680,5.36,0,105,54.0,1
1590,1.0,48.9,22.230,6.31,0,241,73.0,1


In [62]:
for i in range(CU_HCC["Derivation_or_Validation_group"].count()):
    mid = math.pow(10,(CU_HCC.loc[i,"logHBVDNA"]))
    CU_HCC.loc[i,"logDNAumol"] = math.log10(mid*5.6)
CU_HCC

,Derivation_or_Validation_group,albuminR,t-bil,logHBVDNA,HCC,HCCdurationwks,age,Cirrhosis,logDNAumol
0,1.0,42.0,18.981,4.77,0,221,69.0,1,5.518188
1,2.0,41.0,13.680,7.83,0,594,75.0,1,8.578188
2,1.0,43.4,17.100,6.93,0,298,41.0,1,7.678188
3,1.0,41.0,3.420,5.47,0,549,47.0,1,6.218188
4,1.0,43.2,15.390,6.52,0,212,68.0,1,7.268188
...,...,...,...,...,...,...,...,...,...
1587,2.0,34.0,10.260,3.88,1,106,64.0,1,4.628188
1588,2.0,29.0,290.700,3.52,0,693,56.0,1,4.268188
1589,1.0,39.0,13.680,5.36,0,105,54.0,1,6.108188
1590,1.0,48.9,22.230,6.31,0,241,73.0,1,7.058188


In [63]:
for i in range(CU_HCC["Derivation_or_Validation_group"].count()):
    if(CU_HCC.loc[i,"Derivation_or_Validation_group"] != 0):
        score = 0
        if (CU_HCC.loc[i,"age"]>50):
            score+=3
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"albuminR"]<=35):
            score+=20
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"t-bil"]>18):
            score+=1.5
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"logDNAumol"]<=4):
            pass
        elif(CU_HCC.loc[i,"logDNAumol"]<=6 and CU_HCC.loc[i,"logDNAumol"]>4):
            score+=1
        else:
            score+=4
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"Cirrhosis"]==1):
            score+=15
        else:
            pass
        CU_HCC.loc[i,"CU_HCC_score"] = score
CU_HCC

,Derivation_or_Validation_group,albuminR,t-bil,logHBVDNA,HCC,HCCdurationwks,age,Cirrhosis,logDNAumol,CU_HCC_score
0,1.0,42.0,18.981,4.77,0,221,69.0,1,5.518188,20.5
1,2.0,41.0,13.680,7.83,0,594,75.0,1,8.578188,22.0
2,1.0,43.4,17.100,6.93,0,298,41.0,1,7.678188,19.0
3,1.0,41.0,3.420,5.47,0,549,47.0,1,6.218188,19.0
4,1.0,43.2,15.390,6.52,0,212,68.0,1,7.268188,22.0
...,...,...,...,...,...,...,...,...,...,...
1587,2.0,34.0,10.260,3.88,1,106,64.0,1,4.628188,39.0
1588,2.0,29.0,290.700,3.52,0,693,56.0,1,4.268188,40.5
1589,1.0,39.0,13.680,5.36,0,105,54.0,1,6.108188,22.0
1590,1.0,48.9,22.230,6.31,0,241,73.0,1,7.058188,23.5


In [64]:
df["CU-HCC"] =  CU_HCC["CU_HCC_score"]
df

,codenumber,sex1M0F,ETV1TDFTAF2,DM,HTN,NA-naive,got,gpt,t-bil,INRR,...,Derivation_or_Validation_group,HBVDNA1yr,HCCdurationwks,PAGE-B,mPAGE-B,REACH-B,PLAN_B_score,PLAN_B_retrain_score,APA-B,CU-HCC
0,1583,0,1,1,0,0.0,52,31,1.11,1.09,...,1.0,1.0,221,17.0,14.0,12.0,0.355658,0.629871,10.0,20.5
1,537,1,1,0,1,1.0,70,73,0.80,1.35,...,2.0,0.0,594,22.0,16.0,16.0,0.463228,0.647754,6.0,22.0
2,930,1,2,0,0,1.0,50,72,1.00,1.21,...,1.0,1.0,298,16.0,11.0,10.0,0.253839,0.310707,1.0,19.0
3,113,0,2,0,0,1.0,28,38,0.20,1.12,...,1.0,1.0,549,4.0,5.0,8.0,0.197364,0.302011,1.0,19.0
4,1049,1,2,0,0,1.0,97,166,0.90,2.13,...,1.0,1.0,212,20.0,14.0,16.0,0.388369,0.412258,3.0,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,130,1,1,0,0,1.0,77,427,0.60,1.03,...,2.0,1.0,106,14.0,15.0,13.0,0.267926,0.686385,6.0,39.0
1588,165,1,1,0,1,1.0,260,236,17.00,2.75,...,2.0,1.0,693,21.0,17.0,12.0,0.389855,0.689201,7.0,40.5
1589,1462,1,1,0,0,1.0,38,44,0.80,1.20,...,1.0,1.0,105,21.0,15.0,11.0,0.447126,0.618932,8.0,22.0
1590,981,1,1,1,1,1.0,51,77,1.30,0.98,...,1.0,1.0,241,22.0,16.0,14.0,0.358544,0.633538,4.0,23.5


In [65]:
df.to_csv("./compare/67_33ModelmPRLrAU_score.csv")

In [66]:
#trainset c-index test
trainmask = (CU_HCC["Derivation_or_Validation_group"]==1)
CU_HCC_train = CU_HCC[trainmask]
CU_HCC_train.loc[:,'HCC'] = CU_HCC_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_train['HCC'],
    CU_HCC_train['HCCdurationwks'],
    CU_HCC_train["CU_HCC_score"], 
)
print(round(cindex[0], 3))

0.604


In [67]:
#testset c-index test
testmask = (CU_HCC["Derivation_or_Validation_group"]==2)
CU_HCC_test = CU_HCC[testmask]
CU_HCC_test.loc[:,'HCC'] = CU_HCC_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_test['HCC'],
    CU_HCC_test['HCCdurationwks'],
    CU_HCC_test["CU_HCC_score"], 
)
print(round(cindex[0], 3))

0.627


In [68]:
allmask = (CU_HCC["Derivation_or_Validation_group"]!=0)
CU_HCC_all = CU_HCC[allmask]
CU_HCC_all.loc[:,'HCC'] = CU_HCC_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_all['HCC'],
    CU_HCC_all['HCCdurationwks'],
    CU_HCC_all["CU_HCC_score"], 
)
print(round(cindex[0], 3))

0.612


## Ours1yr

In [72]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAU_score.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks', 'PAGE-B', 'mPAGE-B', 'REACH-B',
       'PLAN_B_score', 'PLAN_B_retrain_score', 'APA-B', 'CU-HCC'],
      dtype='object')

In [73]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['sex1M0F', 'ETV1TDFTAF2', 'HBVDNA1yr', 'age1yr', 'got1yr', 'T-bil1yrR', 'albuminR.1', 'platelet1yrR', 'afp1yrR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("1YR754_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursFirst'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursFirst"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

feature: Index(['sex1M0F', 'ETV1TDFTAF2', 'HBVDNA1yr', 'age1yr', 'got1yr', 'T-bil1yrR',
       'albuminR.1', 'platelet1yrR', 'afp1yrR'],
      dtype='object')
trainset C-index 0.776
testset C-index 0.754
all C-index 0.769


In [74]:
print(df)

      codenumber  sex1M0F  ETV1TDFTAF2  DM  HTN  NA-naive  got  gpt  t-bil   
0           1583        0            1   1    0       0.0   52   31   1.11  \
1            537        1            1   0    1       1.0   70   73   0.80   
2            930        1            2   0    0       1.0   50   72   1.00   
3            113        0            2   0    0       1.0   28   38   0.20   
4           1049        1            2   0    0       1.0   97  166   0.90   
...          ...      ...          ...  ..  ...       ...  ...  ...    ...   
1587         130        1            1   0    0       1.0   77  427   0.60   
1588         165        1            1   0    1       1.0  260  236  17.00   
1589        1462        1            1   0    0       1.0   38   44   0.80   
1590         981        1            1   1    1       1.0   51   77   1.30   
1591         436        0            1   0    1       1.0   32   23   1.20   

      INRR  ...  HBVDNA1yr  HCCdurationwks  PAGE-B  mPAGE-B  RE

In [75]:
df.to_csv("./compare/67_33ModelmPRLrAUF.csv")

## OursALL

In [76]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAUF.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks', 'PAGE-B', 'mPAGE-B', 'REACH-B',
       'PLAN_B_score', 'PLAN_B_retrain_score', 'APA-B', 'CU-HCC', 'oursFirst'],
      dtype='object')

In [77]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['albuminR', 'plateletR', 'age1yr', 'got1yr', 'platelet1yrR', 'afp1yrR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("ALL761_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursALL'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursALL"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

feature: Index(['albuminR', 'plateletR', 'age1yr', 'got1yr', 'platelet1yrR', 'afp1yrR'], dtype='object')
trainset C-index 0.813
testset C-index 0.761
all C-index 0.796


In [78]:
print(df)

      codenumber  sex1M0F  ETV1TDFTAF2  DM  HTN  NA-naive  got  gpt  t-bil   
0           1583        0            1   1    0       0.0   52   31   1.11  \
1            537        1            1   0    1       1.0   70   73   0.80   
2            930        1            2   0    0       1.0   50   72   1.00   
3            113        0            2   0    0       1.0   28   38   0.20   
4           1049        1            2   0    0       1.0   97  166   0.90   
...          ...      ...          ...  ..  ...       ...  ...  ...    ...   
1587         130        1            1   0    0       1.0   77  427   0.60   
1588         165        1            1   0    1       1.0  260  236  17.00   
1589        1462        1            1   0    0       1.0   38   44   0.80   
1590         981        1            1   1    1       1.0   51   77   1.30   
1591         436        0            1   0    1       1.0   32   23   1.20   

      INRR  ...  HCCdurationwks  PAGE-B  mPAGE-B  REACH-B  PLAN

In [79]:
df.to_csv("./compare/67_33ModelmPRLrAUFA.csv")

## OursBASELINE

In [80]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAUFA.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df["age"]=df["age1yr"]-1
df.columns

Index(['codenumber', 'sex1M0F', 'ETV1TDFTAF2', 'DM', 'HTN', 'NA-naive', 'got',
       'gpt', 't-bil', 'INRR', 'albuminR', 'plateletR', 'afpR', 'HBeAg',
       'logHBVDNA', 'age1yr', 'got1yr', 'gpt1yr', 'T-bil1yrR', 'platelet1yrR',
       'afp1yrR', 'albuminR.1', 'HCC', 'Derivation_or_Validation_group',
       'HBVDNA1yr', 'HCCdurationwks', 'PAGE-B', 'mPAGE-B', 'REACH-B',
       'PLAN_B_score', 'PLAN_B_retrain_score', 'APA-B', 'CU-HCC', 'oursFirst',
       'oursALL', 'age'],
      dtype='object')

In [81]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['sex1M0F', 'ETV1TDFTAF2', 'age', 'got', 'gpt', 't-bil', 'albuminR', 'plateletR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("BASELINE709_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursBASELINE'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursBASELINE"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

feature: Index(['sex1M0F', 'ETV1TDFTAF2', 'age', 'got', 'gpt', 't-bil', 'albuminR',
       'plateletR'],
      dtype='object')
trainset C-index 0.798
testset C-index 0.709
all C-index 0.769


In [82]:
print(df)

      codenumber  sex1M0F  ETV1TDFTAF2  DM  HTN  NA-naive  got  gpt  t-bil   
0           1583        0            1   1    0       0.0   52   31   1.11  \
1            537        1            1   0    1       1.0   70   73   0.80   
2            930        1            2   0    0       1.0   50   72   1.00   
3            113        0            2   0    0       1.0   28   38   0.20   
4           1049        1            2   0    0       1.0   97  166   0.90   
...          ...      ...          ...  ..  ...       ...  ...  ...    ...   
1587         130        1            1   0    0       1.0   77  427   0.60   
1588         165        1            1   0    1       1.0  260  236  17.00   
1589        1462        1            1   0    0       1.0   38   44   0.80   
1590         981        1            1   1    1       1.0   51   77   1.30   
1591         436        0            1   0    1       1.0   32   23   1.20   

      INRR  ...  mPAGE-B  REACH-B  PLAN_B_score  PLAN_B_retrain

In [ ]:
df.drop("age",axis=1,inplace=True)
df.to_csv("./compare/67_33ModelmPRLrAUFAB.csv")